In [12]:
from pathlib import Path

from feature import (
    make_stft_extractor
)

from testing import(
    load_model_config,
    load_all_template_features_new,
    load_raven_selection_table,
    get_exclusion_intervals,
    detection_pipeline,
    merge_consecutive_detections
)

In [13]:
RESULTS = Path(r"C:\Users\HP\Desktop\Skripsie data\Results")

DATA_29 = Path(r"C:\Users\HP\Desktop\Skripsie data\29")
RESULTS_29 = DATA_29 / "template_selections"

DATA_22 = Path(r"C:\Users\HP\Desktop\Skripsie data\22")

In [14]:
#editable variables
feat_method = "stft"
n_temps = 5
step_frac = 0.75
min_windows = 1
max_windows = 7
max_window_gap = 1

In [15]:
#download relevent calibration info
cfg = load_model_config(RESULTS / f"model_config_{feat_method}_{n_temps}.json")

feat_method_test = cfg["feature"]["method"]
fs_new = cfg["feature"]["fs_new"]
feat_frame_length = cfg["feature"]["frame_len"]
window_length = cfg["window_len"]
vote_frac = cfg["best_vote_frac"]
n_temps_test = cfg["n_temps"]
n_calib = cfg["n_calib"]

call_thresholds = cfg["thresholds"]      # {"st": ..., "mt": ..., "bt": ...}
st_threshold = call_thresholds["st"]
mt_threshold = call_thresholds["mt"]
bt_threshold = call_thresholds["bt"]

# Template files used during calibration
# st_temp_file = cfg["template_files"]["st"]
# mt_temp_file = cfg["template_files"]["mt"]
# bt_temp_file = cfg["template_files"]["bt"]

# Rebuild the extractor exactly as it was during calibration
if feat_method == "stft":
    extractor = make_stft_extractor(frame_dur=feat_frame_length)

step_length = round(window_length * step_frac,3)
print(step_length)

if feat_method_test == feat_method and n_temps_test == n_temps:
    print("Passed download test")

print(f"{feat_method_test}, fs_new = {fs_new}, frame len = {feat_frame_length}, win len = {window_length}, vote frac = {vote_frac}, n_temps = {n_temps_test}, n_calib = {n_calib}")
print(f"st: {st_threshold}, mt: {mt_threshold}, bt: {bt_threshold}")

0.075
Passed download test
stft, fs_new = 1000, frame len = 0.048, win len = 0.1, vote frac = 0.4, n_temps = 5, n_calib = 10
st: 0.14611168909540684, mt: 0.18780400789384374, bt: 0.14542744763697768


In [16]:
st_feats, mt_feats, bt_feats = load_all_template_features_new(
    results_dir=RESULTS_29,
    file_label="20230429_Templates",
    recording_label="20230429_102841",
    extractor=extractor,
    n_temps=n_temps,
    fs_new=fs_new,
)

In [17]:
ground_truth = load_raven_selection_table(DATA_22 / "2023-04-22T17-13-01_000_00070.txt")
td_intervals = get_exclusion_intervals(ground_truth, exclude_labels=("td",), pad=0.1)

In [18]:
#all_kth
all_detected_t, all_detected_labels = detection_pipeline(
    audio_file=DATA_22 / "20230422_171301.wav",
    n_hours=2,
    st_feats=st_feats, mt_feats=mt_feats, bt_feats=bt_feats,
    st_threshold=st_threshold, mt_threshold=mt_threshold, bt_threshold=bt_threshold,
    exclude_intervals=td_intervals,
    extractor=extractor, 
    fs_new=fs_new, window_len=window_length, step_len = step_length, vote_frac= vote_frac
)

Processing hour 1 of 2: 1111 / 47999 windows flagged as calls
Processing hour 2 of 2: 847 / 47999 windows flagged as calls


In [19]:
ground_truth_eval = [g for g in ground_truth if g["label"] not in ("td",)]

In [20]:
calls = merge_consecutive_detections(
    all_detected_t, 
    all_detected_labels,
    step=step_length, 
    window_len=window_length,
    max_gap_windows = max_window_gap,
    min_windows = min_windows,
    max_windows = max_windows
)
